# Conformer Block

## Import

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

## Convolution Module

In [13]:
class ConvolutionModule(nn.Module):
  def __init__(self, channels, kernel_size=1, dropout=0.1):
    super().__init__()

    self.layer_norm = nn.LayerNorm(channels)

    self.pointwise_conv1 = nn.Conv1d(in_channels=channels,
                                     out_channels=2*channels,
                                     kernel_size=kernel_size,
                                     stride=1,
                                     padding=0)

    self.depthwise_conv = nn.Conv1d(
        in_channels=channels,
        out_channels=channels,
        kernel_size=kernel_size,
        stride=1,
        padding=(kernel_size - 1)//2,
        groups=channels
    )

    self.batch_norm = nn.BatchNorm1d(channels)

    self.pointwise_conv2 = nn.Conv1d(in_channels=channels,
                                     out_channels=channels,
                                     kernel_size=kernel_size,
                                     stride=1,
                                     padding=0)

    self.dropout = nn.Dropout(dropout)

  def swish(self, x):
    return x * torch.sigmoid(x)

  def forward(self, x):
    '''
    input: (N, T, d_model)
    output: (N, T, d_model)
    '''
    # input: (N, T, C)
    # (N, T, C)
    x_prime = x # (N, T, C)
    x = self.layer_norm(x) # (N, T, C)

    # from here change the shape of x to (N,C,T)
    x = x.transpose(1, 2) # (N, C, T)
    x = self.pointwise_conv1(x) # (N, C, T)
    x = F.glu(x, dim=1) # (N, 2*C, T)
    x = self.depthwise_conv(x) # (N, C, T)
    x = self.batch_norm(x) # (N, C, T)
    x = self.swish(x) # (N, C, T)
    x = self.pointwise_conv2(x) # (N, C, T)
    x = self.dropout(x) # (N, C, T)

    # now change x shape back to (N, T, C)
    x = x.transpose(1, 2) # (N, T, C)
    x = x + x_prime
    return x

In [14]:
N, T, C = 1, 100, 32
conv_module = ConvolutionModule(C)
sample_input = torch.randn(N, T, C)
sample_output = conv_module(sample_input)
print(f"sample output shape: {sample_output.shape}")

sample output shape: torch.Size([1, 100, 32])


## FeedForward Module

In [17]:
class FeedForwardModule(nn.Module):
  def __init__(self, d_model, expansion_factor=4, dropout=0.1):
    super().__init__()
    self.layer_norm = nn.LayerNorm(d_model)
    self.linear1 = nn.Linear(d_model, d_model*expansion_factor)
    self.linear2 = nn.Linear(d_model*expansion_factor, d_model)
    self.dropout = nn.Dropout(dropout)


  def swish(self, x):
    return x * torch.sigmoid(x)

  def forward(self, x):
    '''
    input: (N, T, d_model)
    output: (N, T, d_model)
    '''
    # (N, T, d_model)
    x_prime = x

    # (N, T, d_model)
    x = self.layer_norm(x)
    x = self.linear1(x)
    x = self.swish(x)
    x = self.dropout(x)
    x = self.linear2(x)
    x = self.dropout(x)

    x = x + x_prime
    # (N, T, d_model)

    return x

In [18]:
N, T, d_model = 1, 100, 32
ffn_module = FeedForwardModule(C)
sample_input = torch.randn(N, T, d_model)
sample_output = ffn_module(sample_input)
print(f"sample output shape: {sample_output.shape}")

sample output shape: torch.Size([1, 100, 32])


## MHA module

In [24]:
class MHAModule(nn.Module):
  def __init__(self, d_model, num_heads, dropout=0.1):
    super().__init__()
    self.layer_norm = nn.LayerNorm(d_model)
    self.mha = nn.MultiheadAttention(
        embed_dim=d_model,
        num_heads=num_heads,
        dropout=dropout,
        batch_first=True
    )
    self.dropout = nn.Dropout(dropout)


  def forward(self, x, attn_mask=None, key_padding_mask=None):
    '''
    input: (N, T, d_model)
    output: (N, T, d_model)
    '''
    x_prime = x # (N, T, d_model)
    x = self.layer_norm(x) # (N, T, d_model)

    # paper used relative pos encoding but for simplicity used fixed pos encoding
    out, _ = self.mha(
        query=x,
        key=x,
        value=x,
        attn_mask=attn_mask, # (T, T)
        key_padding_mask=key_padding_mask, # (N, T)
        need_weights=False
    )

    out = self.dropout(out)
    out = out + x_prime
    return out

In [23]:
N, T, d_model, num_heads = 1, 100, 32, 8
mha_module = MHAModule(d_model, num_heads)
sample_input = torch.randn(N, T, d_model)
sample_output = mha_module(sample_input)
print(f"sample output shape: {sample_output.shape}")

sample output shape: torch.Size([1, 100, 32])


## Putting all together

In [25]:
class Conformer(nn.Module):
  def __init__(self, embedding_dim, num_heads, dropout=0.1):
    super().__init__()

    self.ffn_1 = FeedForwardModule(embedding_dim)
    self.mha_module = MHAModule(embedding_dim, num_heads)
    self.conv_module = ConvolutionModule(embedding_dim)
    self.ffn_2 = FeedForwardModule(embedding_dim)
    self.layer_norm = nn.LayerNorm(embedding_dim)

  def forward(self, x):
    x_prime = x
    x = self.ffn_1(x)
    x = 0.5*x + x_prime

    x_prime = x
    x = self.mha_module(x)
    x = x + x_prime

    x_prime = x
    x = self.conv_module(x)
    x = x + x_prime

    x_prime = x
    x = self.ffn_2(x)
    x = 0.5*x + x_prime

    x = self.layer_norm(x)

    return x

In [26]:
N, T, d_model, num_heads = 1, 100, 32, 8
conformer = Conformer(d_model, num_heads)
sample_input = torch.randn(N, T, d_model)
sample_output = conformer(sample_input)
print(f"sample output shape: {sample_output.shape}")

sample output shape: torch.Size([1, 100, 32])
